# Anote 1A — Unified Dataset

Combines the code-generation and bug-fixing evaluator files into one dataset (working-order step 3: integration).

**Inputs**
- `data/code_generation/code_generation_eval.jsonl` (from `code_generation_EDA_pipeline.ipynb`)
- `data/bug_fixes/bug_fixing_eval.jsonl` (from `bug_fixes_EDA_pipeline.ipynb`)
- each task's `*_full.parquet`, only to fill the `metadata` column
- `data/code_generation/code_generation_private_tests.jsonl` (the full private-test file, from `code_generation_EDA_pipeline.ipynb`)

**Outputs** (in `data/unified/`)
- `unified_eval.jsonl`: the trimmed, stratified dataset: 100 code-generation tasks (the main experiment) + 140 bug-fixing tasks (secondary). Filter on `task_type` to use one or the other.
- `unified_pool.jsonl`: the verified tasks trimming left out, kept as the retrieval / few-shot pool
- `unified_private_tests.jsonl.gz`: the private (hidden) tests of the 100 code-generation tasks only, keyed by `id`, for final scoring
- `reports/trim_summary.csv`: rows kept per dataset and stratum

**Steps**
1. Add a `metadata` column: task-specific details, one dict per row
2. Keep only the columns both task types share (everything else stays in each task's `*_full.parquet`)
3. Concatenate and check every row is tagged with its `task_type`
4. Drop malformed rows
5. Deduplicate
6. Trim each dataset to a stratified sample (code generation: only tasks with private tests)
7. Export
8. Copy the evaluation set's private tests

## 1. Setup

In [4]:
import json
import re
from pathlib import Path

import pandas as pd

# Repo root = first folder at or above the working directory that contains Evaluator/evaluator.py
REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Evaluator" / "evaluator.py").exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Run this notebook from inside the repo (Colab: git clone the repo and %cd into notebooks/).")

DATA_DIR = REPO_ROOT / "data"
INPUTS = {
    "code_generation": DATA_DIR / "code_generation" / "code_generation_eval.jsonl",
    "bug_fixing": DATA_DIR / "bug_fixes" / "bug_fixing_eval.jsonl",
}
FULL_FILES = {
    "code_generation": DATA_DIR / "code_generation" / "code_generation_full.parquet",
    "bug_fixing": DATA_DIR / "bug_fixes" / "bug_fixing_full.parquet",
}
PROCESSED_DIR = DATA_DIR / "unified"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("[INFO] Environment initialized. Directories verified.")

[INFO] Environment initialized. Directories verified.


## 2. Load both task types
`dtype=False` keeps `tests` as real lists (and ids as strings).

In [5]:
frames = {task: pd.read_json(path, lines=True, dtype=False) for task, path in INPUTS.items()}

for task, df in frames.items():
    print(f"[INFO] {task}: {len(df)} rows, columns = {list(df.columns)}")

[INFO] code_generation: 551 rows, columns = ['id', 'task_type', 'source', 'prompt', 'reference', 'tests', 'test_format', 'entry_point']
[INFO] bug_fixing: 204 rows, columns = ['id', 'task_type', 'source', 'prompt', 'reference', 'tests', 'test_format', 'entry_point']


## 3. Add per-task metadata
Every row gets a `metadata` dict, so both task types still share the same columns. The keys depend on the task type:

| `task_type` | `metadata` keys |
|---|---|
| `code_generation` | `difficulty` (`easy` / `medium` / `hard`), `difficulty_method` (`static_complexity` for MBPP, `platform` for LiveCodeBench; the two scales are not comparable), `complexity_score` (MBPP only), `lcb_platform` (`atcoder` / `leetcode`, LiveCodeBench only), `num_private_tests` (hidden tests for this task in `data/code_generation/code_generation_private_tests.jsonl`; 0 = none) |
| `bug_fixing` | `buggy_code`, `bug_diff` (the reference fix as a diff), `num_changed_lines`, `bug_type` (HumanEval-Fix only) |

**Never send `metadata` to the model.** `bug_diff` is the answer. The model only ever sees `prompt`. The metadata is for scoring and analysis:
- Diff the model's output against `buggy_code` to see which lines it changed, then compare with `bug_diff`: did it fix the right place, even when tests fail?
- Split results by `num_changed_lines`: 180 of 204 bug fixes are one-line (all of QuixBugs, 140 of 164 HumanEval-Fix). This is the column that records it.

In [6]:
METADATA_COLS = {
    "code_generation": ["difficulty", "difficulty_method", "complexity_score", "lcb_platform", "num_private_tests"],
    "bug_fixing": ["buggy_code", "bug_diff", "num_changed_lines", "bug_type"],
}

for task, df in frames.items():
    full = pd.read_parquet(FULL_FILES[task]).set_index("id")[METADATA_COLS[task]]
    missing = set(df["id"]) - set(full.index)
    assert not missing, f"{task}: ids missing from {FULL_FILES[task].name}: {sorted(missing)[:5]}"

    # NaN -> None so missing values export as JSON null
    full = full.astype(object).where(full.notna(), None)
    records = full.to_dict("index")
    df["metadata"] = df["id"].map(records)

print(frames["bug_fixing"]["metadata"].iloc[0])

{'buggy_code': 'def bitcount(n):\n    count = 0\n    while n:\n        n ^= n - 1\n        count += 1\n    return count', 'bug_diff': '--- buggy\n+++ fixed\n@@ -4 +4 @@\n-        n ^= n - 1\n+        n &= n - 1', 'num_changed_lines': 1, 'bug_type': None}


## 4. Keep only the shared columns
Any column that isn't in **both** files is dropped. The column order follows the code-generation file.

In [7]:
shared_cols = [c for c in frames["code_generation"].columns if all(c in df.columns for df in frames.values())]
for task, df in frames.items():
    dropped = [c for c in df.columns if c not in shared_cols]
    print(f"{task}: dropping {dropped if dropped else 'nothing'}")

df_unified = pd.concat([df[shared_cols] for df in frames.values()], ignore_index=True)

# Every row must be tagged with one of the known task types
assert set(df_unified["task_type"]) == set(INPUTS), f"Unexpected task_type values: {set(df_unified['task_type'])}"
print(f"\nShared columns: {shared_cols}")
print(f"Total rows: {len(df_unified)}")

code_generation: dropping nothing
bug_fixing: dropping nothing

Shared columns: ['id', 'task_type', 'source', 'prompt', 'reference', 'tests', 'test_format', 'entry_point', 'metadata']
Total rows: 755


## 5. Drop malformed rows
A row is malformed if it can't be sent to a model or scored:
- `id`, `task_type`, `source` or `prompt` is missing / empty
- `tests` is not a non-empty list
- `reference` is empty, **except** for LiveCodeBench, which ships no reference solutions by design (it is scored on tests only)

In [8]:
NO_REFERENCE_SOURCES = {"livecodebench"}


def is_blank(value) -> bool:
    return not isinstance(value, str) or not value.strip()


def malformed_reason(row) -> str:
    for col in ["id", "task_type", "source", "prompt"]:
        if is_blank(row[col]):
            return f"empty {col}"
    if not isinstance(row["tests"], list) or not row["tests"]:
        return "no tests"
    if is_blank(row["reference"]) and row["source"] not in NO_REFERENCE_SOURCES:
        return "empty reference"
    return ""


df_unified["malformed_reason"] = df_unified.apply(malformed_reason, axis=1)
malformed = df_unified[df_unified["malformed_reason"] != ""]
if len(malformed):
    print("Dropped malformed rows:\n" + malformed[["id", "malformed_reason"]].to_string(index=False))
df_unified = df_unified[df_unified["malformed_reason"] == ""].drop(columns="malformed_reason").reset_index(drop=True)
print(f"Dropped {len(malformed)} malformed rows. Remaining: {len(df_unified)}")

Dropped 0 malformed rows. Remaining: 755


## 6. Deduplicate
- `id` must be unique across the whole dataset.
- **Same task, reworded prompt:** two rows of the same `task_type` with the same reference solution (whitespace ignored) are the same problem. The first one is kept. (Example: MBPP 758 and 779 have the same reference; only the prompt wording and one space in a test differ.)
- **Identical prompts** within a task type are also dropped.

In [9]:
assert df_unified["id"].is_unique, "Duplicate ids across task types"


def norm(s) -> str:
    return re.sub(r"\s+", " ", s).strip() if isinstance(s, str) else ""


df_unified["_ref_key"] = df_unified["reference"].map(norm)
df_unified["_prompt_key"] = df_unified["prompt"].map(norm)

has_ref = df_unified["_ref_key"] != ""
dup_problem = has_ref & df_unified.duplicated(["task_type", "_ref_key"])
dup_prompt = df_unified.duplicated(["task_type", "_prompt_key"])
is_dup = dup_problem | dup_prompt

if is_dup.any():
    print("Dropped duplicates:\n" + df_unified.loc[is_dup, ["id", "task_type", "source"]].to_string(index=False))
df_unified = df_unified[~is_dup].drop(columns=["_ref_key", "_prompt_key"]).reset_index(drop=True)
print(f"Dropped {is_dup.sum()} duplicates. Remaining: {len(df_unified)}")

Dropped duplicates:
      id       task_type source
mbpp-779 code_generation   mbpp
Dropped 1 duplicates. Remaining: 754


## 7. Trim each dataset to a stratified sample

Each dataset is cut to a fixed size by **stratified sampling** (fixed seed, so re-running gives the same sample). The code-generation part follows Milestone #2: **20 MBPP + 80 LiveCodeBench = 100 tasks**. Bug fixes stay in as the secondary set; filter on `task_type` to use only code generation.

**Only code-generation tasks with private (hidden) tests can be kept.** The retry loop shows the model the public tests and scores it on the private ones, so a task without private tests can't be scored. All LiveCodeBench tasks have private tests. MBPP tasks that aren't in MBPP+, or whose reference fails MBPP+'s tests, don't, so they go to the pool. The `eligible` column in the summary counts the tasks that could be picked.

| Dataset | Keep | How | Why |
|---|---|---|---|
| MBPP | 20 | 10 `medium` + 10 `hard` (static complexity); the easy third is skipped | Longer, more complex solutions give CodeBLEU / AST similarity something to measure, and fail at attempt 0 more often, so more of them enter the retry loop. The easy third is mostly 1–3 line one-liners |
| LiveCodeBench | 80 | 27 `easy` + 27 `medium` + 26 `hard` (platform labels), AtCoder / LeetCode proportional within each tier | The difficulty axis for the experiment, balanced across tiers |
| QuixBugs | all 40 | — | Secondary (bug fixing). Already small |
| HumanEval-Fix | 100 | bug type × one-line vs multi-line, proportional | Secondary (bug fixing). Keeps all 6 bug types and the ~15% multi-line fixes |

Result: 240 tasks (code generation 100, bug fixing 140). Change `TRIM_PLAN` to resize.

Trimmed-out rows go to `unified_pool.jsonl` for retrieval / few-shot. A dataset set to keep `0` goes to neither file. (HumanEval is no longer loaded at all: the code-generation notebook dropped it, since it shares problems and references with HumanEval-Fix.)

In [10]:
SEED = 42

# source -> (rows to keep, stratify-by columns)
#   int:  that many rows, each stratum keeping its share
#   dict: a fixed count per value of the first stratify-by column, proportional within it for the rest
#   None keeps every row; 0 drops the dataset
TRIM_PLAN = {
    "mbpp": ({"medium": 10, "hard": 10}, ["difficulty"]),
    "livecodebench": ({"easy": 27, "medium": 27, "hard": 26}, ["difficulty", "lcb_platform"]),
    "quixbugs": (None, []),
    "humanevalfix": (100, ["bug_type", "multi_line"]),
}
missing_plan = set(df_unified["source"]) - set(TRIM_PLAN)
assert not missing_plan, f"No trim rule for: {missing_plan}"

# Stratum features, computed per row
meta = df_unified["metadata"]
features = pd.DataFrame({
    "difficulty": meta.map(lambda m: m.get("difficulty")),
    "lcb_platform": meta.map(lambda m: m.get("lcb_platform")),
    "bug_type": meta.map(lambda m: m.get("bug_type")),
    "multi_line": meta.map(lambda m: (m.get("num_changed_lines") or 0) > 1),
    # Code generation is scored on private tests, so only tasks that have them can be kept
    "eligible": (df_unified["task_type"] != "code_generation") | meta.map(lambda m: (m.get("num_private_tests") or 0) > 0),
}, index=df_unified.index)


def stratified_sample(df: pd.DataFrame, n: int, strata: list) -> pd.DataFrame:
    """Proportional allocation per stratum (largest remainder), then a seeded sample inside each stratum."""
    if n is None or n >= len(df):
        return df
    key = features.loc[df.index, strata].astype(str).agg(" | ".join, axis=1) if strata else pd.Series("all", index=df.index)
    sizes = key.value_counts()
    quota = sizes / sizes.sum() * n
    alloc = quota.astype(int)
    alloc[(quota - alloc).sort_values(ascending=False).index[: n - alloc.sum()]] += 1
    return pd.concat([df[key == k].sample(alloc[k], random_state=SEED) for k in alloc.index if alloc[k] > 0])


def trim_source(df: pd.DataFrame, n_keep, strata: list) -> pd.DataFrame:
    """Applies one TRIM_PLAN rule (see the comment above TRIM_PLAN)."""
    if not isinstance(n_keep, dict):
        return stratified_sample(df, n_keep, strata)
    first, rest = strata[0], strata[1:]
    parts = []
    for value, n in n_keep.items():
        group = df[features.loc[df.index, first].astype(str) == value]
        assert len(group) >= n, f"Only {len(group)} {first}={value!r} rows, but TRIM_PLAN asks for {n}"
        parts.append(stratified_sample(group, n, rest))
    return pd.concat(parts)


kept, summary = [], []
for source, df_source in df_unified.groupby("source", sort=False):
    n_keep, strata = TRIM_PLAN[source]
    # Ineligible rows are never sampled, but still go to the pool below
    eligible = df_source[features.loc[df_source.index, "eligible"]]
    sample = trim_source(eligible, n_keep, strata) if n_keep != 0 else df_source.iloc[0:0]
    kept.append(sample)
    summary.append({"source": source, "verified": len(df_source), "eligible": len(eligible), "kept": len(sample),
                    "stratified_by": ", ".join(strata) or "-"})

kept_ids = set(pd.concat(kept)["id"])
dropped_sources = [s for s, (n, _) in TRIM_PLAN.items() if n == 0]
df_pool = df_unified[~df_unified["id"].isin(kept_ids) & ~df_unified["source"].isin(dropped_sources)].reset_index(drop=True)
df_trimmed = df_unified[df_unified["id"].isin(kept_ids)].reset_index(drop=True)  # keeps the original row order

trim_summary = pd.DataFrame(summary)
print(trim_summary.to_string(index=False))
print(f"\nTrimmed dataset: {len(df_trimmed)} tasks | pool: {len(df_pool)} | dropped datasets: {dropped_sources}")

# Strata check: each stratum's share in the sample vs the verified and eligible sets
for source, (n_keep, strata) in TRIM_PLAN.items():
    if strata and n_keep:
        rows = df_unified["source"] == source
        print(f"\n--- {source} ---")
        print(pd.DataFrame({
            "verified": features[rows].groupby(strata).size(),
            "eligible": features[rows & features["eligible"]].groupby(strata).size(),
            "kept": features[rows & df_unified["id"].isin(kept_ids)].groupby(strata).size(),
        }).fillna(0).astype(int).to_string())

       source  verified  eligible  kept            stratified_by
         mbpp       376       288    20               difficulty
livecodebench       174       174    80 difficulty, lcb_platform
     quixbugs        40        40    40                        -
 humanevalfix       164       164   100     bug_type, multi_line

Trimmed dataset: 240 tasks | pool: 514 | dropped datasets: []

--- mbpp ---
            verified  eligible  kept
difficulty                          
easy             154       134     0
hard             124        86    10
medium            98        68    10

--- livecodebench ---
                         verified  eligible  kept
difficulty lcb_platform                          
easy       atcoder             26        26    16
           leetcode            17        17    11
hard       atcoder             60        60    20
           leetcode            20        20     6
medium     atcoder             25        25    13
           leetcode            26       

## 8. Export

In [11]:
REPORTS_DIR = PROCESSED_DIR / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

unified_path = PROCESSED_DIR / "unified_eval.jsonl"
pool_path = PROCESSED_DIR / "unified_pool.jsonl"
df_trimmed.to_json(unified_path, orient="records", lines=True)
df_pool.to_json(pool_path, orient="records", lines=True)
trim_summary.to_csv(REPORTS_DIR / "trim_summary.csv", index=False)

# Round-trip: tests must come back as lists, metadata as dicts
for path, df in [(unified_path, df_trimmed), (pool_path, df_pool)]:
    check = pd.read_json(path, lines=True, dtype=False)
    assert len(check) == len(df) and isinstance(check.loc[0, "tests"], list)
    assert check["metadata"].map(lambda m: isinstance(m, dict)).all(), "Every row needs a metadata dict"
assert not set(df_trimmed["id"]) & set(df_pool["id"]), "A task is in both the dataset and the pool"

print("=== Unified Dataset (trimmed) ===")
print(df_trimmed.groupby(["task_type", "source"]).size().to_string())
print(f"Total tasks: {len(df_trimmed)}")
print(f"\n[SUCCESS] Saved to {PROCESSED_DIR.resolve()}:")
print(f"          - {unified_path.name}: {len(df_trimmed)} tasks ({len(shared_cols)} columns)")
print(f"          - {pool_path.name}: {len(df_pool)} tasks (retrieval / few-shot pool)")
print(f"          - reports/trim_summary.csv")

=== Unified Dataset (trimmed) ===
task_type        source       
bug_fixing       humanevalfix     100
                 quixbugs          40
code_generation  livecodebench     80
                 mbpp              20
Total tasks: 240

[SUCCESS] Saved to /Users/diadana/Documents/GitHub/Anote-1A-building-high-quality-datasets-and-evaluation-frameworks-for-ai-coding-assistants/data/unified:
          - unified_eval.jsonl: 240 tasks (9 columns)
          - unified_pool.jsonl: 514 tasks (retrieval / few-shot pool)
          - reports/trim_summary.csv


## 9. Private tests for the evaluation set

The code-generation notebook writes the **full** private-test file: every verified MBPP and LiveCodeBench task, about 190 MB, git-ignored. Scoring only needs the tasks in `unified_eval.jsonl`, so this step copies just those into `unified_private_tests.jsonl.gz`, about 28 MB.

- One row per code-generation task in `unified_eval.jsonl`: `id` (the same id as in `unified_eval.jsonl`) and `private_test_cases` (same format as that task's `tests`). Bug fixing has no public/private split, so it has no rows.
- Every evaluation task must have private tests. Trimming only picks tasks that have them (section 7), and this step checks it again.
- It is gzipped, with a fixed timestamp so re-running without changes gives the same bytes. Read it with `pd.read_json(path, lines=True, dtype=False)` (pandas unzips `.gz` itself) or `gzip.open(path, "rt")`.
- Never send it to the model. It is only for final scoring.

In [ ]:
import gzip

PRIVATE_TESTS_FULL = DATA_DIR / "code_generation" / "code_generation_private_tests.jsonl"  # from code_generation_EDA_pipeline.ipynb
private_path = PROCESSED_DIR / "unified_private_tests.jsonl.gz"
eval_ids = set(df_trimmed.loc[df_trimmed["task_type"] == "code_generation", "id"])

# Stream the full file line by line (it is large) and keep only the evaluation set's rows.
# mtime=0 keeps the .gz byte-identical across re-runs, so git only sees real changes.
written = set()
with open(PRIVATE_TESTS_FULL) as full, open(private_path, "wb") as raw, \
        gzip.GzipFile(fileobj=raw, mode="wb", mtime=0) as gz:
    for line in full:
        row = json.loads(line)
        if "id" not in row:
            raise KeyError(f"{PRIVATE_TESTS_FULL.name} has no `id` column: re-run code_generation_EDA_pipeline.ipynb")
        if row["id"] in eval_ids:
            gz.write((json.dumps({"id": row["id"], "private_test_cases": row["private_test_cases"]}) + "\n").encode())
            written.add(row["id"])

missing = eval_ids - written
assert not missing, f"{len(missing)} evaluation tasks have no private tests: {sorted(missing)[:5]}"

# Round-trip: one row per evaluation task, each with a non-empty list of tests
check = pd.read_json(private_path, lines=True, dtype=False)
assert len(check) == len(eval_ids) and set(check["id"]) == eval_ids
assert check["private_test_cases"].map(lambda t: isinstance(t, list) and len(t) > 0).all()

n_tests = check.assign(source=check["id"].str.split("-").str[0], n=check["private_test_cases"].map(len))
print(n_tests.groupby("source").agg(tasks=("id", "size"), private_tests=("n", "sum")).to_string())
print(f"\n[SUCCESS] {private_path.name}: {len(check)} tasks ({private_path.stat().st_size / 1e6:,.1f} MB, "
      f"from the {PRIVATE_TESTS_FULL.stat().st_size / 1e6:,.0f} MB full file)")